# 01 — Acquire

Downloads GRETIL Sanskrit plaintext files and the DCS CoNLL-U corpus.
Writes `data/raw/PROVENANCE.txt` with URLs, commit hash, download date, and license info.

In [ ]:
import os, re, subprocess, time
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

BASE  = Path('/Users/sidharthbildikar/Desktop/code/sanskrit_corpus')
RAW   = BASE / 'data' / 'raw'
GRETIL_DIR = RAW / 'gretil'
DCS_DIR    = RAW / 'dcs'

for d in [RAW, GRETIL_DIR, DCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

GRETIL_BASE = 'https://gretil.sub.uni-goettingen.de'
DCS_REPO    = 'https://github.com/OliverHellwig/sanskrit'
TODAY       = datetime.now(timezone.utc).strftime('%Y-%m-%d')
print('Setup complete. TODAY =', TODAY)

## GRETIL: scrape index and collect Sanskrit plaintext URLs

In [ ]:
r = requests.get(f'{GRETIL_BASE}/gretil.html', timeout=30)
r.raise_for_status()
soup = BeautifulSoup(r.text, 'lxml')

links = [a['href'] for a in soup.find_all('a', href=True)]

# Only want Sanskrit plaintext files: path contains 'plaintext' and ends with .txt
# Files are under gretil/corpustei/transformations/plaintext/sa_*.txt
sa_txt = sorted(set(
    l for l in links
    if 'plaintext' in l and l.endswith('.txt') and '/sa_' in l
))

print(f'Sanskrit plaintext files found: {len(sa_txt)}')
print('Sample URLs:')
for u in sa_txt[:5]:
    print(' ', u)

## GRETIL: download all .txt files (throttled, 4 workers)

In [ ]:
_sess = requests.Session()
_sess.headers['User-Agent'] = 'sanskrit-corpus-builder/1.0 (academic research)'

def _dl_gretil(rel_path):
    url  = f'{GRETIL_BASE}/{rel_path}'
    dest = GRETIL_DIR / Path(rel_path).name
    if dest.exists():
        return ('skip', dest.name)
    try:
        resp = _sess.get(url, timeout=60)
        resp.raise_for_status()
        dest.write_bytes(resp.content)
        time.sleep(0.3)  # polite pacing per worker
        return ('ok', dest.name)
    except Exception as exc:
        return ('err', f'{url}: {exc}')

errors  = []
skipped = 0
ok      = 0

with ThreadPoolExecutor(max_workers=4) as pool:
    futs = {pool.submit(_dl_gretil, p): p for p in sa_txt}
    for fut in tqdm(as_completed(futs), total=len(futs), desc='GRETIL download'):
        status, info = fut.result()
        if status == 'err':
            errors.append(info)
        elif status == 'skip':
            skipped += 1
        else:
            ok += 1

gretil_files = sorted(GRETIL_DIR.glob('sa_*.txt'))
print(f'\nGRETIL: {ok} downloaded, {skipped} already cached, {len(errors)} errors')
print(f'Files on disk: {len(gretil_files)}')

if errors:
    print('\nERRORS (first 10):')
    for e in errors[:10]:
        print(' ', e)
    if len(errors) > 10:
        print(f'  ... and {len(errors)-10} more')

## DCS: sparse git clone (only `dcs/data/conllu/files`)

In [ ]:
dcs_repo_dir = DCS_DIR / 'sanskrit'

if not (dcs_repo_dir / '.git').exists():
    print('Cloning DCS repo (sparse, depth 1, no blobs) — this may take a minute...')
    subprocess.run([
        'git', 'clone',
        '--depth', '1',
        '--filter=blob:none',
        '--sparse',
        DCS_REPO,
        str(dcs_repo_dir)
    ], check=True)
else:
    print('DCS repo already cloned')

# Narrow sparse checkout to just the conllu files directory
print('Setting sparse-checkout path to dcs/data/conllu/files ...')
subprocess.run(
    ['git', 'sparse-checkout', 'set', 'dcs/data/conllu/files'],
    cwd=str(dcs_repo_dir), check=True
)

# Verify blobs are present (git fetches them lazily on checkout)
conllu_root = dcs_repo_dir / 'dcs' / 'data' / 'conllu' / 'files'
conllu_files = list(conllu_root.rglob('*.conllu'))
print(f'CoNLL-U files on disk: {len(conllu_files)}')

if len(conllu_files) == 0:
    print('\nWARNING: no .conllu files found after sparse-checkout.')
    print('Attempting explicit checkout to trigger blob fetch...')
    subprocess.run(['git', 'checkout'], cwd=str(dcs_repo_dir))
    conllu_files = list(conllu_root.rglob('*.conllu'))
    print(f'After checkout: {len(conllu_files)} files')

assert len(conllu_files) > 0, 'STOP: DCS CoNLL-U files not found after clone.'
print(f'Sample: {conllu_files[0]}')

## Record DCS commit hash

In [ ]:
dcs_commit = subprocess.run(
    ['git', 'rev-parse', 'HEAD'],
    cwd=str(dcs_repo_dir), capture_output=True, text=True, check=True
).stdout.strip()

print(f'DCS commit: {dcs_commit}')

## Write PROVENANCE.txt

In [ ]:
provenance = f"""Sanskrit Corpus — Data Provenance
Generated: {TODAY}
====================================

SOURCE 1: GRETIL (Göttingen Register of Electronic Texts in Indian Languages)
  URL          : https://gretil.sub.uni-goettingen.de/gretil.html
  Subdirectory : gretil/corpustei/transformations/plaintext/
  Files        : {len(gretil_files)} Sanskrit plaintext files (sa_*.txt)
  Download date: {TODAY}
  License      : Individual files carry Creative Commons Attribution-NonCommercial-ShareAlike
                 4.0 International (CC BY-NC-SA 4.0) unless otherwise noted.
                 See each file's '## Licence:' header for per-file details.
  Citation     : GRETIL — Göttingen Register of Electronic Texts in Indian Languages.
                 Niedersächsische Staats- und Universitätsbibliothek Göttingen.
                 https://gretil.sub.uni-goettingen.de/

SOURCE 2: Digital Corpus of Sanskrit (DCS)
  URL          : https://github.com/OliverHellwig/sanskrit
  Subdirectory : dcs/data/conllu/files/
  Git commit   : {dcs_commit}
  Files        : {len(conllu_files)} CoNLL-U files
  Download date: {TODAY}
  License      : Creative Commons Attribution 4.0 International (CC BY 4.0).
                 See https://github.com/OliverHellwig/sanskrit/blob/master/LICENSE
  Citation     : Oliver Hellwig and Sebastian Nehrdich.
                 Sanskrit Word Segmentation Using Character-level Recurrent and
                 Convolutional Neural Networks. EMNLP 2018.
                 https://aclanthology.org/D18-1295/

NOTE ON DATA USE
  This corpus is assembled for non-commercial academic research.
  The DCS CoNLL-U files are kept intact in data/raw/dcs/sanskrit/dcs/data/conllu/files/
  as required by downstream annotation tasks.
"""

(RAW / 'PROVENANCE.txt').write_text(provenance, encoding='utf-8')
print('PROVENANCE.txt written')
print(provenance)